# MODUL PRAKTIKUM BIG DATA 2
## Pemrosesan Terstruktur Menggunakan PySpark DataFrame API & Spark SQL

**Nama:** Alya Maura Raditha  
**NIM:** 52240003

Modul praktikum kedua ini berfokus pada pemrosesan data terstruktur skala besar menggunakan **Spark DataFrame API** dan **Spark SQL**. Praktikum ini menggunakan **NYC Yellow Taxi Trips Dataset** untuk melatih kemampuan data ingestion, transformasi waktu, agregasi, segmentasi kueri SQL, dan *window functions*.

---

### 📌 Dataset yang Digunakan
* **Nama Dataset:** NYC Yellow Taxi Trip Data
* **Deskripsi:** Dataset transaksi perjalanan taksi New York City yang memuat informasi lokasi/jam *pickup*, jarak tempuh, durasi, tarif, dan tips.

### 📝 Catatan Adaptasi ke Google Colab
Notebook ini dijalankan di **Google Colab**. Bagian yang bersifat khusus Databricks disesuaikan sebagai berikut:

| Komponen pada modul asli (Databricks) | Penyesuaian pada notebook ini (Colab) |
|---|---|
| `%sh wget ...` dan `cp ... /dbfs/FileStore/tables/` | Dataset dibaca dari Google Drive (`/content/drive/My Drive/data/yellow_tripdata_2015-01.csv`), disalin ke `/content/data/` |
| Variabel `spark` tersedia otomatis | `SparkSession` dibuat manual (mode `local[*]`) |
| `dbfs:/FileStore/tables/nyc_taxi.csv` | `/content/data/nyc_taxi.csv` |
| `dbfs:/FileStore/tables/output/nyc_hourly_summary.parquet` | `/content/output/nyc_hourly_summary.parquet` |

**Penting – Menjalankan notebook:** gunakan menu **Runtime → Run all**. Hasil `print()`/`show()` setiap tahapan direkam otomatis dan disimpan ke file **`/content/output_2.txt`** pada sel paling akhir.

### 🔎 Catatan tentang Interpretasi
Setiap sel analisis diikuti sel **Interpretasi Hasil** yang ditulis berdasarkan hasil eksekusi nyata pada dataset `yellow_tripdata_2015-01.csv` (12,748,986 baris). Seluruh angka bersifat deterministik dan akan sama saat dijalankan ulang. Satuan uang pada dataset ini adalah **dolar AS (USD)**.

Analisis mengikuti instruksi tugas apa adanya. Bila interpretasi menyebut adanya distorsi data (mis. nilai jarak yang tidak masuk akal), itu disajikan sebagai **catatan analitis**; kode tidak diubah dari ketentuan tugas.

## 0. Persiapan Lingkungan Google Colab

In [ ]:
# ==============================================================================
# 0. PERSIAPAN LINGKUNGAN GOOGLE COLAB
# ==============================================================================
# Instal PySpark. Versi 3.5.3 dipin agar perilaku fungsi (mis. to_timestamp, cast)
# konsisten di seluruh runtime Colab.
!pip install -q pyspark==3.5.3 

import os, sys, io, shutil, subprocess, time, datetime

# Spark membutuhkan Java. Colab umumnya sudah menyediakannya; jika belum, pasang otomatis.
if shutil.which("java") is None:
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"], check=True)

print("Java tersedia :", shutil.which("java") is not None)
print("Versi Python  :", sys.version.split()[0])

In [ ]:
# ------------------------------------------------------------------------------
# Perekam output: setiap hasil print() juga disimpan ke LOG_BUFFER, lalu di sel
# terakhir ditulis ke file output_2.txt (output aktual hasil eksekusi notebook ini).
# ------------------------------------------------------------------------------
class _Tee:
    def __init__(self, stream, buf):
        self._stream, self._buf = stream, buf
    def write(self, s):
        self._buf.write(s)
        return self._stream.write(s)
    def flush(self):
        self._stream.flush()
    def __getattr__(self, name):
        return getattr(self._stream, name)

LOG_BUFFER = io.StringIO()
if not isinstance(sys.stdout, _Tee):
    sys.stdout = _Tee(sys.stdout, LOG_BUFFER)

def banner(title):
    """Cetak judul bagian agar file output tersusun sistematis sesuai urutan tahapan."""
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

RUN_STARTED = datetime.datetime.now()

## 1. Pengambilan Dataset NYC Taxi dari Google Drive

Dataset **`yellow_tripdata_2015-01.csv`** dibaca dari Google Drive pada folder **`/content/drive/My Drive/data`**. Saat sel dijalankan, Colab akan meminta izin akses Drive.

Karena file ini besar (sekitar 1,9 GB), file **disalin dulu ke disk lokal Colab** (`/content/data/nyc_taxi.csv`) agar pemrosesan Spark jauh lebih cepat daripada membaca langsung dari Drive. Penyalinan hanya dilakukan sekali.

In [ ]:
banner("1. PENGAMBILAN DATASET NYC TAXI DARI GOOGLE DRIVE")

# Mount Google Drive (di luar Colab, langkah ini dilewati dan folder yang sudah ada dipakai apa adanya)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("(Bukan Google Colab: mount Drive dilewati.)")

DRIVE_DIR = "/content/drive/My Drive/data"
DRIVE_FILE = f"{DRIVE_DIR}/yellow_tripdata_2015-01.csv"

if not os.path.exists(DRIVE_FILE):
    isi = sorted(os.listdir(DRIVE_DIR)) if os.path.isdir(DRIVE_DIR) else "(folder tidak ditemukan)"
    raise FileNotFoundError(
        f"File tidak ditemukan: {DRIVE_FILE}\n"
        f"Isi folder {DRIVE_DIR}: {isi}\n"
        "Pastikan Drive sudah terhubung dan nama file/folder persis sama (huruf besar-kecil berpengaruh)."
    )

LOCAL_DIR = "/content/data"
os.makedirs(LOCAL_DIR, exist_ok=True)
CSV_PATH = f"{LOCAL_DIR}/nyc_taxi.csv"           # salinan lokal yang dibaca Spark

# Salin ke disk lokal Colab (dilewati jika salinan lengkap sudah ada)
if os.path.exists(CSV_PATH) and os.path.getsize(CSV_PATH) == os.path.getsize(DRIVE_FILE):
    print("Salinan lokal sudah lengkap; penyalinan dilewati.")
else:
    t0 = time.time()
    shutil.copyfile(DRIVE_FILE, CSV_PATH)
    print(f"Penyalinan selesai dalam {time.time() - t0:.1f} detik.")

print(f"Sumber (Drive)  : {DRIVE_FILE}  ({os.path.getsize(DRIVE_FILE)/1e6:,.2f} MB)")
print(f"Salinan lokal   : {CSV_PATH}  ({os.path.getsize(CSV_PATH)/1e6:,.2f} MB)")
print("3 baris pertama file:")
with open(CSV_PATH, encoding="utf-8", errors="replace") as f:
    for _ in range(3):
        print("  ", f.readline().rstrip())

## 2. Ingestion Data dengan Explicit Schema

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Spark session (di Databricks otomatis tersedia sebagai `spark`; di Colab dibuat manual)
spark = (SparkSession.builder
         .master("local[*]")
         .appName("PraktikumBigData2_DataFrame_SQL")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.ansi.enabled", "false")      # nilai yang gagal di-cast menjadi NULL (tidak menghentikan proses)
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")   # string tanggal yang tak cocok pola -> NULL, bukan error
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

# Skema eksplisit untuk dataset NYC Taxi
schema = StructType([
    StructField("vendor_id", StringType(), True),
    StructField("pickup_datetime", StringType(), True),
    StructField("dropoff_datetime", StringType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("rate_code_id", StringType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("payment_type", StringType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])

file_path = f"file://{CSV_PATH}"

banner("2. INGESTION DATA DENGAN EXPLICIT SCHEMA")

# --- Pemeriksaan header file terhadap skema eksplisit -------------------------
# Spark memetakan skema eksplisit berdasarkan URUTAN kolom, bukan nama kolom. Karena itu header file
# diperiksa dulu: bila sama dengan skema -> baca langsung dengan skema eksplisit (seperti modul asli);
# bila berbeda urutan/penamaan -> kolom dipetakan berdasarkan NAMA lalu di-cast ke tipe pada skema.
EXPECTED = [f.name for f in schema.fields]
ALIASES = {"vendorid": "vendor_id",
           "tpep_pickup_datetime": "pickup_datetime", "lpep_pickup_datetime": "pickup_datetime",
           "tpep_dropoff_datetime": "dropoff_datetime", "lpep_dropoff_datetime": "dropoff_datetime",
           "ratecodeid": "rate_code_id", "rate_code": "rate_code_id"}
norm = lambda c: ALIASES.get(c.strip().lower(), c.strip().lower())

raw_header = spark.read.option("header", "true").csv(file_path).columns
header_norm = [norm(c) for c in raw_header]
print("Kolom pada file :", raw_header)

if header_norm[:len(EXPECTED)] == EXPECTED:
    mode_baca = "posisional dengan skema eksplisit (header sesuai skema)"
    df_raw = (spark.read.format("csv")
              .option("header", "true")
              .schema(schema)
              .load(file_path))
else:
    missing = [c for c in EXPECTED if c not in header_norm]
    if missing:
        raise ValueError(f"Kolom berikut dibutuhkan skema tetapi tidak ada pada file: {missing}")
    mode_baca = "berdasarkan nama kolom, lalu cast ke tipe skema eksplisit (header berbeda urutan/penamaan)"
    df_str = spark.read.option("header", "true").csv(file_path).toDF(*header_norm)
    df_raw = df_str.select([F.col(f.name).cast(f.dataType).alias(f.name) for f in schema.fields])
print("Mode pembacaan  :", mode_baca)

print("\n--- SKEMA SPARK DATAFRAME ---")
df_raw.printSchema()
n_raw = df_raw.count()
print(f"Total Record Data: {n_raw:,}")

## 3. Pembersihan Data & Feature Engineering

In [ ]:
banner("3. PEMBERSIHAN DATA & FEATURE ENGINEERING")

# Cleaning & Datetime Extraction
df_clean = df_raw.dropna(subset=["pickup_datetime", "total_amount"]) \
    .filter((F.col("trip_distance") > 0) & (F.col("total_amount") > 0))

# Format tanggal utama sesuai modul ("yyyy-MM-dd HH:mm:ss"). Sebagai pengaman, akhiran zona waktu " UTC"
# (lazim pada ekspor data taksi) dibuang terlebih dahulu, dan to_timestamp tanpa pola dipakai sebagai cadangan.
TS_FMT = "yyyy-MM-dd HH:mm:ss"

def parse_ts(col_name):
    cleaned = F.regexp_replace(F.trim(F.col(col_name)), r"\s*UTC$", "")
    return F.coalesce(F.to_timestamp(cleaned, TS_FMT), F.to_timestamp(cleaned))

df_transformed = df_clean \
    .withColumn("pickup_ts", parse_ts("pickup_datetime")) \
    .withColumn("dropoff_ts", parse_ts("dropoff_datetime")) \
    .withColumn("pickup_hour", F.hour("pickup_ts")) \
    .withColumn("trip_duration_min", F.round((F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 60, 2))

# Dataset besar (belasan juta baris) diproses berkali-kali -> simpan hasil transformasi (RAM, tumpah ke disk bila perlu)
from pyspark import StorageLevel
df_transformed.persist(StorageLevel.MEMORY_AND_DISK)

# Mendaftarkan Temporary View untuk Kueri Spark SQL
df_transformed.createOrReplaceTempView("v_nyc_taxi")

# Pemeriksaan kualitas hasil parsing waktu (harus tanpa NULL agar analisis per jam valid)
n_clean = df_transformed.count()
n_null_pickup = df_transformed.filter(F.col("pickup_ts").isNull()).count()
n_null_dropoff = df_transformed.filter(F.col("dropoff_ts").isNull()).count()
n_nonpos_dur = df_transformed.filter(F.col("trip_duration_min") <= 0).count()
print(f"Record setelah pembersihan (trip_distance > 0 & total_amount > 0): {n_clean:,} dari {n_raw:,}")
print(f"Pickup timestamp gagal di-parse (NULL)  : {n_null_pickup:,}")
print(f"Dropoff timestamp gagal di-parse (NULL) : {n_null_dropoff:,}")
print(f"Perjalanan dengan durasi <= 0 menit     : {n_nonpos_dur:,}  (informasi kualitas data; tidak difilter)")
assert n_clean > 0, "Tidak ada data tersisa setelah pembersihan."
assert n_null_pickup < n_clean, "Seluruh pickup_datetime gagal di-parse; periksa format tanggal pada dataset."

print("\n--- CONTOH DATA TERTATA (5 BARIS) ---")
df_transformed.select("vendor_id", "pickup_hour", "trip_distance", "trip_duration_min", "total_amount").show(5)

#### 🔎 Interpretasi Hasil — Tahap 1-3 (Pengambilan, Ingestion, dan Pembersihan Data)

**Temuan utama**
- File sumber berukuran **1,985.96 MB** dengan **12,748,986 baris**; penyalinan dari Drive ke disk lokal Colab memerlukan 38.7 detik. Header file bergaya TLC (`VendorID`, `tpep_pickup_datetime`, `RateCodeID`, dst.), sehingga notebook otomatis memakai mode **pemetaan berdasarkan nama kolom** lalu mengonversi ke tipe skema eksplisit. Lima kolom yang tidak ada pada skema modul (koordinat pickup/dropoff dan `improvement_surcharge`) diabaikan.
- Setelah filter `trip_distance > 0` dan `total_amount > 0` (serta pembuangan nilai kosong), tersisa **12,666,418 baris (99.35%)**; 82,568 baris (0.65%) tersingkir.
- Parsing waktu **100% berhasil**: 0 `pickup_ts` dan 0 `dropoff_ts` bernilai NULL. Hanya **880 perjalanan (0.0069%)** berdurasi ≤ 0 menit.

**Makna**
- Kualitas **format** data sangat baik (tanggal, tipe, kelengkapan), sehingga seluruh analisis per jam valid. Namun kualitas **isi** data tidak sebersih formatnya: filter minimum hanya membuang nilai nol/negatif, bukan nilai ekstrem yang tidak masuk akal. Tugas A1 dan C1 di bawah memperlihatkan dampaknya pada rata-rata dan nilai maksimum.
- Tahap 4 (contoh demonstrasi) menghasilkan tabel yang identik dengan Tugas A1, sehingga interpretasinya digabung pada Tugas A1.

## 4. Agregasi DataFrame API (Contoh Demonstrasi)

In [ ]:
banner("4. AGREGASI DATAFRAME API (CONTOH DEMONSTRASI)")

df_hourly_summary = df_transformed.groupBy("pickup_hour") \
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    ) \
    .sort(F.desc("total_trips"))

print("--- RINGKASAN PERJALANAN PER JAM ---")
df_hourly_summary.show(5)

## 5. TUGAS MANDIRI MAHASISWA (SPARK DATAFRAME & SPARK SQL)

Kerjakan tugas-tugas di bawah ini menggunakan **Spark DataFrame API** atau **Spark SQL** pada sel kode di bawah!

**Dataset:** NYC Taxi Trips Dataset (`v_nyc_taxi` / `df_transformed`)

### KATEGORI A: ANALISIS TREN WAKTU & METRIK PERJALANAN (DATAFRAME API)

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS A1: Analisis Jam Sibuk Penumpang (Peak Pick-up Hours)
# Instruksi:
# 1. Kelompokkan data berdasarkan `pickup_hour` (Jam 00 s.d. 23).
# 2. Hitung:
#    - Total frekuensi perjalanan (Count)
#    - Rata-rata jarak tempuh (`trip_distance`)
#    - Total pendapatan (`total_amount`)
# 3. Urutkan berdasarkan total frekuensi perjalanan terbanyak dan tampilkan Top 5 Jam Sibuk!
# ------------------------------------------------------------------------------
banner("TUGAS A1 — TOP 5 JAM SIBUK PENUMPANG (PEAK PICK-UP HOURS)")

# 1-2. Group by pickup_hour + tiga metrik. Seluruh jam (00-23) dipertahankan pada df_a1_summary
#      karena DataFrame ini juga dipakai untuk ekspor Parquet pada Tugas C2.
df_a1_summary = (df_transformed
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    )
    # 3. Urut dari frekuensi terbanyak; pickup_hour sebagai pemutus seri agar hasil deterministik
    .orderBy(F.desc("total_trips"), F.asc("pickup_hour"))
    .cache())

print(f"Jumlah jam (pickup_hour) berbeda pada data: {df_a1_summary.count()}")
print("\n--- TOP 5 JAM SIBUK PENUMPANG ---")
df_a1_summary.limit(5).show()

#### 🔎 Interpretasi Hasil — Tugas A1 (Top 5 Jam Sibuk Penumpang)

**Temuan utama**
- Lima jam tersibuk berdasarkan jumlah perjalanan: **19:00 (800,980)**, **18:00 (795,133)**, **20:00 (730,153)**, **21:00 (707,689)**, dan **22:00 (683,197)**. Kelimanya **berurutan (18:00-22:59)** dan bersama-sama **3,717,152 perjalanan = 29.35%** dari 12,666,418, padahal hanya 5 dari 24 jam. Bila tersebar merata, porsinya hanya 20.83%, jadi puncak ini **1.41 kali** lebih padat.
- Jam 19 menyumbang 6.32% perjalanan (**1.52 kali** porsi seragam 4.17%). Jam 19 dan jam 18 nyaris setara (selisih 5,847 perjalanan, 0.74%), lalu volume menurun bertahap hingga jam 22 yang 14.7% lebih rendah dari jam 19.

**Makna**
- Permintaan taksi berpusat pada **sore hingga malam** (jam pulang kerja, makan malam, dan aktivitas malam hari). Implikasinya, ketersediaan armada paling kritis pada 18:00-22:00, dan jam 18-19 adalah rentang dengan beban tertinggi.
- Peringkat berdasarkan **jumlah perjalanan tetap dapat dipercaya**, karena nilai ekstrem mengubah jarak dan tarif, bukan jumlah baris.

**Peringatan penting: `avg_distance` dan `total_revenue` pada tabel ini terdistorsi outlier**
- Rata-rata jarak 25.87 mil (jam 19), 41.91 (jam 21), dan 34.80 (jam 22) tidak masuk akal sebagai rata-rata satu perjalanan taksi kota. Buktinya ada pada Tugas C1: satu perjalanan pada jam 18 tercatat sejauh **2,331,800 mil**. Perjalanan itu sendiri menambah sekitar **2.93 mil** ke rata-rata jam 18; tanpanya rata-rata jam 18 turun dari 6.10 menjadi sekitar **3.2 mil**.
- Total jarak jam 19, 21, dan 22 masing-masing setara sekitar 20.7 juta, 29.7 juta, dan 23.8 juta mil. Dengan rata-rata wajar sekitar 3.2 mil, jam 19 seharusnya hanya sekitar 2.5 juta mil. Artinya sebagian besar total jarak berasal dari **segelintir catatan jarak yang salah**, bukan dari perjalanan nyata.
- Pendapatan per perjalanan: jam 18 USD 14.37, jam 20 USD 14.55, jam 21 USD 15.03, jam 22 USD 15.36, tetapi **jam 19 USD 19.18**. Selisih sekitar USD 3.8-4.8 per perjalanan itu setara kelebihan sekitar **USD 3.1-3.9 juta** pada jam 19, mengindikasikan adanya nilai `total_amount` ekstrem. Dengan kata lain, keunggulan jam 19 pada `total_revenue` (USD 15.36 juta dibanding USD 11.43 juta pada jam 18) kemungkinan besar **bukan keunggulan nyata**.
- Untuk analisis yang andal, gunakan median/persentil atau saring jarak dan tarif yang tidak masuk akal. Angka `1.536148889E7` dan sejenisnya hanyalah format tampilan ilmiah Spark untuk 15,361,488.89.

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS A2: Analisis Durasi Perjalanan Berdasarkan Jumlah Penumpang
# Instruksi:
# 1. Kelompokkan data berdasarkan `passenger_count`.
# 2. Hitung rata-rata durasi perjalanan (`trip_duration_min`) dan rata-rata total tarif (`total_amount`).
# 3. Filter kelompok penumpang yang lebih dari 0 dan urutkan berdasarkan jumlah penumpang.
# ------------------------------------------------------------------------------
banner("TUGAS A2 — DURASI & TARIF RATA-RATA BERDASARKAN JUMLAH PENUMPANG")

df_a2 = (df_transformed
    .filter(F.col("passenger_count") > 0)                      # 3. hanya kelompok penumpang > 0
    .groupBy("passenger_count")                                # 1. kelompokkan per jumlah penumpang
    .agg(                                                      # 2. rata-rata durasi & total tarif
        F.round(F.avg("trip_duration_min"), 2).alias("avg_trip_duration_min"),
        F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
        F.count("*").alias("total_trips")                      # kolom tambahan: ukuran tiap kelompok
    )
    .orderBy("passenger_count"))                               # 3. urut berdasarkan jumlah penumpang

df_a2.show()

#### 🔎 Interpretasi Hasil — Tugas A2 (Durasi & Tarif Berdasarkan Jumlah Penumpang)

**Temuan utama**
- Kelompok 1-6 penumpang memuat hampir seluruh data: 12,660,203 perjalanan berpenumpang > 0, sedangkan 6,215 perjalanan (0.049%) berpenumpang 0 tersaring. Perjalanan **1 penumpang mendominasi 70.52%** (8,927,494), diikuti 2 penumpang (14.26%), 5 penumpang (5.49%), 3 penumpang (4.16%), 6 penumpang (3.58%), dan 4 penumpang (1.99%).
- Rata-rata durasi untuk kelompok 1-6 hanya berkisar **13.80-14.55 menit** (selisih 5.4%), dan rata-rata total tarif **USD 14.63-15.31** (selisih 4.6%). Durasi terlama ada pada 5 penumpang (14.55 menit), tarif tertinggi pada 2 penumpang (USD 15.31).
- Kelompok 7, 8, dan 9 penumpang hanya berisi **7, 3, dan 6 perjalanan** (total 16, 0.00013%) dengan rata-rata yang sangat berbeda (tarif USD 17.05, 39.17, dan 84.88; durasi 11.32, 17.41, dan 28.87 menit).

**Makna**
- **Jumlah penumpang hampir tidak memengaruhi durasi maupun tarif.** Selisih antarkelompok utama hanya sekitar 5%, jadi durasi dan tarif ditentukan oleh jarak, lalu lintas, dan waktu, bukan oleh banyaknya penumpang (taksi dikenai tarif per perjalanan, bukan per penumpang).
- Rata-rata kelompok 7-9 penumpang **tidak dapat diandalkan** karena hanya berasal dari 3-7 perjalanan; satu perjalanan panjang saja mengubah rata-ratanya drastis. Jumlah penumpang sebanyak itu juga melebihi kapasitas taksi kuning pada umumnya, sehingga kemungkinan besar merupakan kesalahan input atau kendaraan khusus.
- Pola jumlah yang tidak lazim: perjalanan 5 penumpang (695,449) dan 6 penumpang (453,124) jauh lebih banyak daripada 4 penumpang (252,105). Ini hanya observasi; data tidak memuat jenis kendaraan sehingga penyebabnya tidak dapat dipastikan.

### KATEGORI B: ANALISIS SQL & SEGMENTASI TARIF (SPARK SQL)

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS B1: Segmentasi Jarak Tempuh Menggunakan CASE WHEN di Spark SQL
# Instruksi:
# Tuliskan kueri SQL dengan `spark.sql()` untuk:
# 1. Mengategorikan `trip_distance` ke dalam 3 kategori:
#    - 'Short Trip' : trip_distance <= 2.0
#    - 'Medium Trip': trip_distance BETWEEN 2.1 AND 10.0
#    - 'Long Trip'  : trip_distance > 10.0
# 2. Hitung jumlah total perjalanan, rata-rata ongkos (`fare_amount`), dan rata-rata tips (`tip_amount`) per segmen!
# ------------------------------------------------------------------------------
banner("TUGAS B1 — SEGMENTASI JARAK TEMPUH (CASE WHEN, SPARK SQL)")

# CATATAN KETEPATAN BATAS: definisi 'BETWEEN 2.1 AND 10.0' pada instruksi menyisakan celah (2.0 < jarak < 2.1),
# misalnya 2.05 mil, yang tidak masuk kategori mana pun (hasilnya NULL). Agar setiap perjalanan tetap
# terklasifikasi, batas bawah Medium ditulis 'trip_distance > 2.0' (maksud yang sama: lanjutan dari Short).
# Cek di bawah menunjukkan berapa baris yang terdampak pada data ini.
celah = spark.sql("SELECT COUNT(*) AS n FROM v_nyc_taxi WHERE trip_distance > 2.0 AND trip_distance < 2.1").first()["n"]
print(f"Perjalanan dengan jarak di celah (2.0 < jarak < 2.1) : {celah:,}  -> tetap dimasukkan ke 'Medium Trip'\n")

df_b1 = spark.sql("""
    SELECT
        CASE
            WHEN trip_distance <= 2.0                    THEN 'Short Trip'
            WHEN trip_distance >  2.0 AND trip_distance <= 10.0 THEN 'Medium Trip'
            WHEN trip_distance > 10.0                    THEN 'Long Trip'
        END                                  AS distance_segment,
        COUNT(*)                             AS total_trips,
        ROUND(AVG(fare_amount), 2)           AS avg_fare_amount,
        ROUND(AVG(tip_amount), 2)            AS avg_tip_amount
    FROM v_nyc_taxi
    GROUP BY 1
    ORDER BY CASE distance_segment WHEN 'Short Trip' THEN 1 WHEN 'Medium Trip' THEN 2 ELSE 3 END
""")
df_b1.show()

# Validasi: seluruh perjalanan harus masuk tepat satu segmen
total_segmen = df_b1.agg(F.sum("total_trips")).first()[0]
assert total_segmen == n_clean, f"Jumlah segmen ({total_segmen}) != jumlah data ({n_clean})"
print(f"Validasi: total perjalanan seluruh segmen = {total_segmen:,} = jumlah data bersih ({n_clean:,}).")

#### 🔎 Interpretasi Hasil — Tugas B1 (Segmentasi Jarak Tempuh dengan CASE WHEN)

**Temuan utama**
- **Short Trip (≤ 2.0 mil): 7,491,192 perjalanan (59.14%)**, tarif rata-rata USD 7.00, tip rata-rata USD 0.90. **Medium Trip: 4,587,365 (36.22%)**, tarif USD 15.59, tip USD 2.88. **Long Trip (> 10 mil): 587,861 (4.64%)**, tarif USD 44.88, tip USD 5.92. Total ketiganya 12,666,418, sama dengan jumlah data bersih (validasi terpenuhi).
- **133,370 perjalanan (1.05%)** berjarak di celah 2.0-2.1 mil. Bila memakai `BETWEEN 2.1 AND 10.0` apa adanya, perjalanan ini akan bernilai NULL dan hilang dari seluruh segmen, sehingga batas bawah Medium ditulis `> 2.0`.

**Makna**
- Hampir 6 dari 10 perjalanan adalah perjalanan pendek, tetapi **pendapatan tarif justru terpusat pada segmen menengah**. Perkiraan kontribusi `fare_amount` (jumlah perjalanan × rata-rata tarif): Short ≈ **34.9%**, Medium ≈ **47.6%**, Long ≈ **17.5%**. Segmen Long hanya 4.64% perjalanan tetapi menyumbang sekitar seperenam pendapatan tarif.
- Tarif naik seiring jarak: Medium 2.23 kali Short, Long 2.88 kali Medium (6.41 kali Short). Tip rata-rata naik dengan pola serupa (USD 0.90, 2.88, 5.92).
- **Rasio tip terhadap tarif** (rata-rata tip dibagi rata-rata tarif): Short ≈ 12.9%, **Medium ≈ 18.5%**, Long ≈ 13.2%. Segmen Medium memiliki tip paling tinggi secara proporsional. Perlu diingat bahwa tip tunai tidak tercatat pada data (lihat Tugas B2), sehingga perbedaan rasio antarsegmen dapat ikut dipengaruhi oleh perbedaan proporsi pembayaran tunai pada tiap segmen.

**Catatan**
- Segmen Long dipengaruhi catatan jarak yang tidak masuk akal (Tugas C1), tetapi tarif rata-ratanya (USD 44.88) tetap wajar. Itu mengindikasikan bahwa nilai jarak ekstrem hanya terjadi pada sangat sedikit baris dan tidak banyak mengubah angka tarif segmen.

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS B2: Analisis Metode Pembayaran & Tip Rate
# Instruksi:
# Tuliskan kueri SQL untuk:
# 1. Kelompokkan data berdasarkan `payment_type`.
# 2. Hitung rata-rata persentase tip (`(tip_amount / total_amount) * 100`).
# 3. Tampilkan tipe pembayaran yang memberikan rata-rata tip tertinggi!
# ------------------------------------------------------------------------------
banner("TUGAS B2 — METODE PEMBAYARAN & TIP RATE (SPARK SQL)")

# 1-2. Rata-rata persentase tip per payment_type (total_amount > 0 sudah dijamin oleh tahap pembersihan)
df_b2 = spark.sql("""
    SELECT
        payment_type,
        COUNT(*)                                          AS total_trips,
        ROUND(AVG((tip_amount / total_amount) * 100), 2)  AS avg_tip_pct
    FROM v_nyc_taxi
    GROUP BY payment_type
    ORDER BY avg_tip_pct DESC
""")
print("--- RATA-RATA TIP RATE (%) PER TIPE PEMBAYARAN (URUT TERTINGGI) ---")
df_b2.show()

# 3. Tipe pembayaran dengan rata-rata tip tertinggi
df_b2_top = spark.sql("""
    SELECT payment_type, ROUND(AVG((tip_amount / total_amount) * 100), 2) AS avg_tip_pct
    FROM v_nyc_taxi
    GROUP BY payment_type
    ORDER BY AVG((tip_amount / total_amount) * 100) DESC
    LIMIT 1
""")
print("--- TIPE PEMBAYARAN DENGAN RATA-RATA TIP TERTINGGI ---")
df_b2_top.show()
top_row = df_b2_top.first()
print(f"Kesimpulan (berdasarkan hasil di atas): payment_type '{top_row['payment_type']}' memberikan "
      f"rata-rata tip tertinggi sebesar {top_row['avg_tip_pct']:.2f}% dari total tarif.")

#### 🔎 Interpretasi Hasil — Tugas B2 (Metode Pembayaran & Tip Rate)

**Temuan utama**
- Tipe pembayaran **1** memiliki rata-rata tip tertinggi, **15.15%** dari `total_amount`, pada 7,853,509 perjalanan (**62.00%** data). Tipe **2** (4,775,278 perjalanan, **37.70%**) bernilai **0.00%**; tipe 3 (29,117; 0.23%) 0.01%; tipe 4 (8,513; 0.07%) 0.02%; tipe 5 hanya 1 perjalanan dengan 0.00%. Total semua tipe 12,666,418.
- Rata-rata tip seluruh perjalanan sekitar **9.4%** (62.00% × 15.15%), karena hampir 38% perjalanan tercatat tanpa tip.

**Makna: ini lebih mencerminkan cara data dicatat daripada perilaku penumpang**
- Menurut kamus data TLC, kode 1 adalah kartu kredit, 2 tunai, 3 tanpa biaya, 4 sengketa, dan 5 tidak diketahui. Pada data TLC, **tip hanya tercatat otomatis untuk pembayaran kartu; tip tunai tidak tercatat**. Karena itu 0.00% pada tipe 2 hampir pasti berarti "tip tunai tidak terekam", **bukan** "penumpang tunai tidak memberi tip".
- Dengan demikian kesimpulan "tipe 1 memberi tip tertinggi" benar secara angka tetapi tidak dapat dipakai untuk membandingkan kedermawanan penumpang kartu dan tunai. Untuk perilaku tip yang bermakna, ambil **hanya perjalanan berkartu** sebagai dasar: pada kelompok itu rata-rata tip sekitar 15%, dan angka ini menjadi acuan yang jauh lebih layak.
- Tipe 3 dan 4 (tanpa biaya dan sengketa) hanya 0.30% perjalanan (37,630) dengan tip nyaris nol, sesuai dengan sifatnya sebagai transaksi tidak normal.

**Catatan**
- Persentase dihitung terhadap `total_amount`, yang sudah **memuat tip itu sendiri**, bukan terhadap tarif dasar. Terhadap tarif, persentase tip akan sedikit lebih besar.

### KATEGORI C: ADVANCED WINDOW FUNCTIONS & EXPORT

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS C1: Ranking Perjalanan Terjauh Per Jam Menggunakan Window Function
# Instruksi:
# 1. Gunakan Window Function (`Window.partitionBy("pickup_hour").orderBy(F.desc("trip_distance"))`).
# 2. Berikan ranking (`dense_rank()`) pada setiap perjalanan berdasarkan jarak tempuh di setiap jamnya.
# 3. Tampilkan perjalanan urutan ke-1 (terjauh) untuk setiap jam pada jam 08:00, 12:00, dan 18:00!
# ------------------------------------------------------------------------------
banner("TUGAS C1 — PERJALANAN TERJAUH PER JAM (WINDOW FUNCTION dense_rank)")
from pyspark.sql.window import Window

# 1. Definisi window: partisi per jam, urut jarak tempuh menurun
w_hour = Window.partitionBy("pickup_hour").orderBy(F.desc("trip_distance"))

# 2. dense_rank() untuk setiap perjalanan pada jamnya
df_ranked = df_transformed.withColumn("distance_rank", F.dense_rank().over(w_hour))

# 3. Perjalanan urutan ke-1 pada jam 08:00, 12:00, dan 18:00
#    (bila ada beberapa perjalanan dengan jarak maksimum yang sama, semuanya ditampilkan karena sama-sama rank 1)
df_c1 = (df_ranked
    .filter((F.col("distance_rank") == 1) & (F.col("pickup_hour").isin(8, 12, 18)))
    .select("pickup_hour", "distance_rank", "vendor_id", "pickup_datetime", "dropoff_datetime",
            "trip_distance", "trip_duration_min", "fare_amount", "total_amount")
    .orderBy("pickup_hour", "pickup_datetime"))
df_c1.show(truncate=False)

#### 🔎 Interpretasi Hasil — Tugas C1 (Perjalanan Terjauh per Jam dengan `dense_rank`)

**Temuan utama**
- Perjalanan terjauh: **jam 08:00** (25 Jan 2015) **196.8 mil** dalam 39.97 menit dengan tarif USD 2.50; **jam 12:00** (4 Jan 2015) **66,746.8 mil** dalam 19.35 menit dengan tarif USD 17.50; **jam 18:00** (3 Jan 2015) **2,331,800 mil** dalam 9.0 menit dengan tarif USD 8.00. Tiap jam hanya muncul satu baris (tidak ada seri di peringkat 1).
- Kecepatan implisit (jarak ÷ durasi): sekitar **295 mph**, **207 ribu mph**, dan **15.5 juta mph**. Jauh melampaui batas fisik kendaraan darat.

**Makna: ketiganya adalah kesalahan pencatatan, bukan perjalanan nyata**
- Ketiga nilai bertentangan dengan kolom lain pada baris yang sama. Tarif USD 2.50, USD 17.50, dan USD 8.00 sesuai perjalanan pendek (USD 2.50 tampak setara tarif awal argo), dan durasi hanya 9-40 menit. Jarak yang tercatat kemungkinan salah karena kegagalan argo/GPS atau kesalahan desimal.
- Hasil ini menjelaskan distorsi pada Tugas A1: satu catatan seperti ini cukup untuk menggeser rata-rata jarak satu jam sebesar beberapa mil. Jadi `MAX(trip_distance)` **tidak informatif** untuk mencari perjalanan terjauh yang nyata pada data tanpa penyaringan.
- Rekomendasi untuk analisis lanjutan: saring jarak ke rentang masuk akal (mis. batas atas ratusan mil), atau tambahkan syarat kecepatan implisit yang realistis (mis. di bawah 100 mph), atau gunakan persentil ke-99 sebagai ganti nilai maksimum. Kode tidak diubah karena instruksi hanya meminta peringkat jarak per jam.

In [ ]:
# ------------------------------------------------------------------------------
# TUGAS C2: Export Hasil Agregasi ke Parquet Format
# Instruksi:
# 1. Simpan DataFrame hasil agregasi jam sibuk (Tugas A1) ke direktori DBFS dalam format terkompresi **Parquet**.
# 2. Lokasi target: `dbfs:/FileStore/tables/output/nyc_hourly_summary.parquet`
# 3. Mode penyimpanan: 'overwrite'.
# ------------------------------------------------------------------------------
banner("TUGAS C2 — EXPORT HASIL AGREGASI A1 KE PARQUET")

# Lokasi target: di Colab, pengganti dbfs:/FileStore/tables/output/ adalah /content/output/
PARQUET_PATH = "/content/output/nyc_hourly_summary.parquet"

# 1-3. Simpan df_a1_summary (agregasi jam sibuk, seluruh jam 00-23) -> Parquet terkompresi (snappy), mode overwrite.
#      coalesce(1) dipakai agar hasilnya satu file data (24 baris saja) dan direktori tetap rapi.
(df_a1_summary.coalesce(1)
    .write.mode("overwrite")
    .option("compression", "snappy")
    .parquet(f"file://{PARQUET_PATH}"))

# Verifikasi: file benar-benar terbentuk dan dapat dibaca kembali
print(f"Direktori Parquet : {PARQUET_PATH}")
for nama in sorted(n for n in os.listdir(PARQUET_PATH) if not n.startswith(".")):   # sembunyikan file .crc internal
    ukuran = os.path.getsize(os.path.join(PARQUET_PATH, nama))
    print(f"  - {nama:<60} {ukuran:>8,} byte")

df_check = spark.read.parquet(f"file://{PARQUET_PATH}")
n_back = df_check.count()
assert n_back == df_a1_summary.count(), "Jumlah baris Parquet tidak sama dengan DataFrame sumber!"
assert any(n.endswith(".parquet") for n in os.listdir(PARQUET_PATH)), "File data parquet tidak ditemukan!"
print(f"\nVerifikasi baca ulang: {n_back} baris terbaca (sama dengan DataFrame agregasi A1).")
print("Skema Parquet:")
df_check.printSchema()
print("Isi Parquet (5 baris teratas berdasarkan jam sibuk):")
df_check.orderBy(F.desc("total_trips"), F.asc("pickup_hour")).show(5)

#### 🔎 Interpretasi Hasil — Tugas C2 (Export Hasil Agregasi A1 ke Parquet)

**Temuan utama**
- Direktori `/content/output/nyc_hourly_summary.parquet` terbentuk berisi satu berkas data berformat snappy (**1,882 byte**) dan penanda `_SUCCESS` (0 byte, tanda penulisan selesai tanpa gagal).
- Baca ulang berhasil: **24 baris** (satu per jam, 00-23), sama dengan DataFrame A1. Skema tersimpan dengan tipe yang benar: `pickup_hour` (integer), `total_trips` (long), `avg_distance` dan `total_revenue` (double). Lima baris teratas identik dengan hasil Tugas A1.

**Makna**
- Parquet menyimpan tipe data dan terkompresi, sehingga 24 baris ringkasan hanya memerlukan sekitar 1.9 KB dan dapat dibaca ulang tanpa kehilangan tipe (berbeda dengan CSV). Pemakaian `coalesce(1)` menghasilkan satu berkas data yang rapi untuk data sekecil ini.
- Perlu diingat bahwa berkas ini memuat nilai **apa adanya** dari Tugas A1, termasuk `avg_distance` dan `total_revenue` yang terdistorsi outlier (lihat interpretasi A1). Pengguna hilir sebaiknya memakai kolom `total_trips`, yang andal, dan berhati-hati terhadap dua kolom lainnya.

## 🧭 Ringkasan Insight Keseluruhan (Penugasan 2)

1. **Kualitas data:** format sangat baik (99.35% baris lolos, parsing waktu 100% berhasil, hanya 880 perjalanan berdurasi ≤ 0), tetapi **isi data memuat outlier ekstrem** (mis. jarak 2,331,800 mil dalam 9 menit) yang lolos filter minimum.
2. **Jam sibuk:** permintaan memuncak pada **18:00-22:59** (29.35% perjalanan dari 5 jam; jam 19 tersibuk dengan 800,980 perjalanan). Peringkat ini andal karena berbasis jumlah perjalanan.
3. **Distorsi rata-rata dan total:** `avg_distance` (hingga 41.91 mil) dan `total_revenue` jam 19 (USD 19.18 per perjalanan dibanding 14.4-15.4 pada jam sekitarnya) dipengaruhi catatan yang salah; gunakan median/persentil atau saring outlier sebelum mengambil kesimpulan nilai.
4. **Jumlah penumpang:** 70.52% perjalanan hanya 1 penumpang, dan jumlah penumpang hampir tidak memengaruhi durasi (13.80-14.55 menit) maupun tarif (USD 14.63-15.31). Kelompok 7-9 penumpang (16 perjalanan) tidak dapat diandalkan.
5. **Segmen jarak:** 59.14% perjalanan pendek, tetapi segmen menengah menyumbang sekitar 47.6% pendapatan tarif dan memiliki rasio tip tertinggi (≈ 18.5% dari tarif).
6. **Tip:** tipe pembayaran 1 (kartu) memberi tip 15.15% dibanding 0.00% pada tunai, namun tip tunai tidak tercatat pada data, sehingga perbedaan ini adalah artefak pencatatan, bukan perilaku penumpang.
7. **Ekspor:** ringkasan jam sibuk berhasil diekspor ke Parquet (24 baris, 1,882 byte) dan terverifikasi dapat dibaca ulang.

## 6. Penyimpanan Output ke File TXT
Sel berikut menulis seluruh hasil dari notebook ini (berurutan sesuai tahapan) ke `/content/output_2.txt` dan mengunduhnya.

In [ ]:
# ==============================================================================
# PENYIMPANAN OUTPUT KE FILE TXT
# ==============================================================================
OUTPUT_TXT = "/content/output_2.txt"

header = (
    "OUTPUT PENUGASAN 2 - PYSPARK DATAFRAME API & SPARK SQL (NYC YELLOW TAXI)\n"
    + "Waktu eksekusi   : " + RUN_STARTED.strftime("%Y-%m-%d %H:%M:%S") + "\n"
    + "Versi PySpark    : " + spark.version + "\n"
    + "Versi Python     : " + sys.version.split()[0] + "\n"
    + "Sumber dataset   : " + "Google Drive (" + DRIVE_FILE + ")" + "\n"
    + "=" * 78 + "\n"
)

with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write(header + LOG_BUFFER.getvalue())

# Verifikasi bahwa file benar-benar terbentuk dan dapat dibuka kembali
assert os.path.exists(OUTPUT_TXT) and os.path.getsize(OUTPUT_TXT) > 0, "File output gagal dibuat!"
with open(OUTPUT_TXT, encoding="utf-8") as f:
    isi = f.read()
print(f"File output tersimpan: {OUTPUT_TXT}  ({os.path.getsize(OUTPUT_TXT):,} byte, {isi.count(chr(10)) + 1} baris)")

# Di Google Colab, unduh file ke komputer Anda (diabaikan bila bukan di Colab)
try:
    from google.colab import files
    files.download(OUTPUT_TXT)
except ImportError:
    pass